# Fairness and bias audit for PhysioNet 2019 sepsis models

This notebook is designed to explore **fairness and bias** in sepsis early warning models using the PhysioNet/Computing in Cardiology 2019 data.

It is organized into:

1. Setup and configuration  
2. Helper functions  
3. Data loading and preprocessing  
4. Cohort overview and subgroup audit  
5. Shared-model training and overall evaluation  
6. Performance by age × sex subgroup  
7. Calibration and threshold tradeoff analysis  
8. Process-bias / ablation analysis  
9. Optional subgroup-specific model training  
10. Save outputs

## Main fairness questions

This notebook focuses on questions like:

- Does a single shared model perform differently across age-sex groups?
- Do some groups receive more false alerts or shorter lead times?
- Are disparities reduced when demographic, missingness, or administrative variables are removed?
- Do group-specific models improve subgroup performance relative to one shared model?

## Important notes

- The dataset contains a binary-coded `Gender` field (`0 = Female`, `1 = Male`) rather than a richer gender representation.
- This notebook evaluates fairness in the **observed labels and measurements**; it cannot fully separate model bias from label bias or care-process bias.
- By default, the notebook trains **shared models** on the full training set and evaluates them within subgroups.
- An **optional** section near the end trains subgroup-specific GLMs for direct comparison.

In [1]:
# =========================
# Setup and imports
# =========================

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.notebook import tqdm

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

print("HAS_XGBOOST:", HAS_XGBOOST)
print("HAS_TORCH:", HAS_TORCH)

HAS_XGBOOST: False
HAS_TORCH: True


In [2]:
# =========================
# Configuration
# =========================

SEED = 1
DATA_DIR = Path("./data/")
OUTPUT_DIR = Path("results_fairness_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "SepsisLabel"
ALL_FEATURES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2",
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
    "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
    "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
    "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
    "Fibrinogen", "Platelets", "Age", "Gender", "Unit1", "Unit2",
    "HospAdmTime", "ICULOS"
]

PHYSIOLOGY_FEATURES = [
    c for c in ALL_FEATURES
    if c not in ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
]
DEMOGRAPHIC_ADMIN_FEATURES = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]

TEST_SIZE = 0.30
LIU_LIKE_WINDOW = (-2, -1)
SEPSIS_LABEL_LEAD = 6
SEQ_LEN = 8
EPOCHS = 12
BATCH_SIZE = 128
LR = 1e-3

AGE_BIN_EDGES = [-np.inf, 45, 75, np.inf]
AGE_BIN_LABELS = ["<45", "45-74", "75+"]
GENDER_MAP = {0: "Female", 1: "Male"}

# Optional expensive steps
TRAIN_XGBOOST = HAS_XGBOOST
TRAIN_GRU = HAS_TORCH
TRAIN_GROUP_SPECIFIC_GLM = False   # set True if you want separate GLMs per subgroup
MIN_GROUP_PATIENTS_FOR_SUBGROUP_MODEL = 200

# Calibration settings
CALIBRATION_BINS = 10

print("DATA_DIR:", DATA_DIR.resolve() if DATA_DIR.exists() else DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("TRAIN_XGBOOST:", TRAIN_XGBOOST)
print("TRAIN_GRU:", TRAIN_GRU)
print("TRAIN_GROUP_SPECIFIC_GLM:", TRAIN_GROUP_SPECIFIC_GLM)

DATA_DIR: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/data
OUTPUT_DIR: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/results_fairness_audit
TRAIN_XGBOOST: False
TRAIN_GRU: True
TRAIN_GROUP_SPECIFIC_GLM: False


In [3]:
# =========================
# Helper classes and functions
# =========================

@dataclass
class PatientSummary:
    patient_id: str
    n_rows: int
    onset_idx: Optional[int]
    became_septic: bool


def set_global_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def find_patient_files(data_dir: Path, recursive: bool = True) -> List[Path]:
    if not data_dir.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {data_dir}")

    pattern_func = data_dir.rglob if recursive else data_dir.glob
    files = list(pattern_func("*.psv")) + list(pattern_func("*.csv"))
    files = sorted(f for f in files if f.is_file() and f.stem.startswith("p"))

    if not files:
        raise FileNotFoundError(f"No patient .psv or .csv files found under {data_dir}")

    print(f"Found {len(files)} patient files.")
    print("First few files:")
    for f in files[:5]:
        print(" -", f)
    return files


def load_patient_file(fp: Path) -> pd.DataFrame:
    if fp.suffix.lower() == ".psv":
        df = pd.read_csv(fp, sep="|")
    elif fp.suffix.lower() == ".csv":
        df = pd.read_csv(fp)
    else:
        raise ValueError(f"Unsupported file type: {fp}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"{fp} is missing required target column '{TARGET_COL}'")

    missing_features = [c for c in ALL_FEATURES if c not in df.columns]
    for c in missing_features:
        df[c] = np.nan

    df = df[ALL_FEATURES + [TARGET_COL]].copy()
    df["patient_id"] = fp.stem
    df["t"] = np.arange(len(df), dtype=int)
    return df


def load_all_patients(data_dir: Path) -> Tuple[pd.DataFrame, List[PatientSummary]]:
    files = find_patient_files(data_dir, recursive=True)
    frames = []
    summaries = []

    for fp in tqdm(files, desc="Loading patient files"):
        df = load_patient_file(fp)
        onset_positions = np.where(df[TARGET_COL].fillna(0).values.astype(int) == 1)[0]
        onset_idx = int(onset_positions[0]) if len(onset_positions) > 0 else None
        became_septic = onset_idx is not None
        summaries.append(PatientSummary(fp.stem, len(df), onset_idx, became_septic))
        frames.append(df)

    full = pd.concat(frames, ignore_index=True)
    return full, summaries


def split_patients(
    summaries: List[PatientSummary],
    test_size: float = 0.30,
    seed: int = 42,
) -> Tuple[List[str], List[str]]:
    patient_ids = [s.patient_id for s in summaries]
    labels = [int(s.became_septic) for s in summaries]
    train_ids, test_ids = train_test_split(
        patient_ids,
        test_size=test_size,
        random_state=seed,
        stratify=labels if len(set(labels)) > 1 else None,
    )
    return sorted(train_ids), sorted(test_ids)


def add_missingness_and_deltas(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    out = df.copy()

    for c in tqdm(feature_cols, desc="Adding missingness flags"):
        out[f"{c}_missing"] = out[c].isna().astype(int)

    out = out.sort_values(["patient_id", "t"]).copy()
    grouped = []
    for _, g in tqdm(out.groupby("patient_id"), desc="Computing deltas", total=out["patient_id"].nunique()):
        g = g.copy()
        ff = g[feature_cols].ffill()
        deltas = ff.diff().fillna(0.0)
        deltas.columns = [f"{c}_delta" for c in feature_cols]
        grouped.append(pd.concat([g, deltas], axis=1))

    return pd.concat(grouped, ignore_index=True)


def get_label_start_and_estimated_onset(
    g: pd.DataFrame,
    target_col: str = TARGET_COL,
    label_lead: int = 6,
):
    g = g.sort_values("t").reset_index(drop=True).copy()
    pos = np.where(g[target_col].fillna(0).values.astype(int) == 1)[0]
    if len(pos) == 0:
        return None, None
    label_start_t = int(g.iloc[pos[0]]["t"])
    estimated_onset_t = label_start_t + label_lead
    return label_start_t, estimated_onset_t


def build_proxy_training_rows(
    df: pd.DataFrame,
    patient_ids: List[str],
    event_window: Tuple[int, int] = (-2, -1),
    label_lead: int = 6,
) -> pd.DataFrame:
    lo, hi = event_window
    sub = df[df["patient_id"].isin(patient_ids)].copy()
    out_frames = []

    for pid, g in tqdm(sub.groupby("patient_id"), desc="Building proxy rows", total=sub["patient_id"].nunique()):
        g = g.sort_values("t").reset_index(drop=True).copy()
        _, event_t = get_label_start_and_estimated_onset(g, TARGET_COL, label_lead=label_lead)

        if event_t is None:
            gg = g.copy()
            gg["proxy_label"] = 0
            out_frames.append(gg)
            continue

        start = event_t + lo
        end = event_t + hi
        mask = (g["t"] >= start) & (g["t"] <= end)
        gg = g.loc[mask].copy()
        if len(gg) == 0:
            continue
        gg["proxy_label"] = 1
        out_frames.append(gg)

    if not out_frames:
        raise RuntimeError("No proxy rows generated.")
    return pd.concat(out_frames, ignore_index=True)


def choose_feature_columns(df: pd.DataFrame, base_features: Optional[List[str]] = None) -> List[str]:
    if base_features is None:
        base_features = ALL_FEATURES
    base = [c for c in base_features if c in df.columns]
    missing = [f"{c}_missing" for c in base_features if f"{c}_missing" in df.columns]
    delta = [f"{c}_delta" for c in base_features if f"{c}_delta" in df.columns]
    return base + missing + delta


def assign_age_bin(age_series: pd.Series, edges=AGE_BIN_EDGES, labels=AGE_BIN_LABELS) -> pd.Series:
    return pd.cut(age_series, bins=edges, labels=labels, right=False, include_lowest=True)


def build_patient_demographics(df: pd.DataFrame, summaries: List[PatientSummary]) -> pd.DataFrame:
    patient_demo = (
        df.sort_values(["patient_id", "t"])
        .groupby("patient_id", as_index=False)
        .first()[["patient_id", "Age", "Gender", TARGET_COL, "ICULOS"]]
        .copy()
    )

    patient_demo = patient_demo.merge(
        pd.DataFrame([s.__dict__ for s in summaries])[["patient_id", "became_septic", "n_rows"]],
        on="patient_id",
        how="left",
    )
    patient_demo["Gender_label"] = patient_demo["Gender"].map(GENDER_MAP).fillna("Missing/Other")
    patient_demo["Age_bin"] = assign_age_bin(patient_demo["Age"]).astype(str).replace("nan", "Missing")
    patient_demo["group"] = patient_demo["Gender_label"] + " | " + patient_demo["Age_bin"]
    return patient_demo


def train_glm(X_train: pd.DataFrame, y_train: np.ndarray) -> Pipeline:
    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="liblinear",
            penalty="l1",
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])
    model.fit(X_train, y_train)
    return model


def train_xgboost(X_train: pd.DataFrame, y_train: np.ndarray):
    if not HAS_XGBOOST:
        raise ImportError("xgboost is not installed.")

    pos = max(1, int(np.sum(y_train == 1)))
    neg = max(1, int(np.sum(y_train == 0)))
    scale_pos_weight = neg / pos

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=SEED,
            scale_pos_weight=scale_pos_weight,
            n_jobs=4,
        )),
    ])
    model.fit(X_train, y_train)
    return model


class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


class GRUClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        h = out[:, -1, :]
        return self.fc(h).squeeze(-1)


def build_patient_sequences(
    df: pd.DataFrame,
    patient_ids: List[str],
    feature_cols: List[str],
    seq_len: int = 8,
    event_window: Tuple[int, int] = (-2, -1),
    label_lead: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    sub = df[df["patient_id"].isin(patient_ids)].copy()
    sequences = []
    labels = []

    for pid, g in tqdm(sub.groupby("patient_id"), desc="Building sequences", total=sub["patient_id"].nunique()):
        g = g.sort_values("t").reset_index(drop=True).copy()
        _, event_t = get_label_start_and_estimated_onset(g, TARGET_COL, label_lead=label_lead)

        if event_t is None:
            for end_t in range(seq_len - 1, len(g)):
                block = g.iloc[end_t - seq_len + 1:end_t + 1]
                sequences.append(block[feature_cols].values)
                labels.append(0)
            continue

        start = event_t + event_window[0]
        end = event_t + event_window[1]
        for end_idx in range(len(g)):
            current_t = int(g.iloc[end_idx]["t"])
            if current_t < max(seq_len - 1, start) or current_t > end:
                continue
            block = g.iloc[end_idx - seq_len + 1:end_idx + 1]
            if len(block) == seq_len:
                sequences.append(block[feature_cols].values)
                labels.append(1)

    if not sequences:
        raise RuntimeError("No sequences created.")
    return np.stack(sequences), np.array(labels, dtype=int)


def median_impute_and_scale_for_rnn(train_seq: np.ndarray, test_seq: np.ndarray):
    n_train, seq_len, n_feat = train_seq.shape
    n_test = test_seq.shape[0]
    tr2 = train_seq.reshape(-1, n_feat)
    te2 = test_seq.reshape(-1, n_feat)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    tr2 = scaler.fit_transform(imputer.fit_transform(tr2))
    te2 = scaler.transform(imputer.transform(te2))

    return tr2.reshape(n_train, seq_len, n_feat), te2.reshape(n_test, seq_len, n_feat), imputer, scaler


def train_rnn(X_train, y_train, X_val, y_val, epochs=12, batch_size=128, lr=1e-3):
    if not HAS_TORCH:
        raise ImportError("torch is not installed.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_ds = SequenceDataset(X_train, y_train)
    val_ds = SequenceDataset(X_val, y_val)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = GRUClassifier(input_size=X_train.shape[-1], hidden_size=64, num_layers=1).to(device)

    cls_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_train.astype(int),
    )
    pos_weight = torch.tensor([cls_weights[1] / cls_weights[0]], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_state = None
    best_auc = -np.inf

    for epoch in tqdm(range(epochs), desc="Training GRU"):
        model.train()
        train_losses = []
        for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy()
                val_probs.extend(probs.tolist())
                val_true.extend(yb.numpy().tolist())

        val_auc = roc_auc_score(val_true, val_probs) if len(set(val_true)) > 1 else np.nan
        print(f"Epoch {epoch+1}: train_loss={np.mean(train_losses):.4f}, val_auc={val_auc:.4f}")

        if np.isfinite(val_auc) and val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_rnn(model, X: np.ndarray) -> np.ndarray:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)
    ds = SequenceDataset(X, np.zeros(len(X)))
    dl = DataLoader(ds, batch_size=256, shuffle=False)
    probs = []
    for xb, _ in tqdm(dl, desc="Predicting RNN probabilities"):
        xb = xb.to(device)
        with torch.no_grad():
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy()
        probs.extend(p.tolist())
    return np.array(probs)


def optimal_threshold_from_roc(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    dist = np.sqrt((1 - tpr) ** 2 + fpr ** 2)
    idx = int(np.argmin(dist))
    return {
        "threshold": float(thr[idx]),
        "tpr": float(tpr[idx]),
        "fpr": float(fpr[idx]),
    }


def evaluate_static_model(model, X_test: pd.DataFrame, y_test: np.ndarray, model_name: str) -> Dict[str, float]:
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    thr_info = optimal_threshold_from_roc(y_test, probs)
    preds = (probs >= thr_info["threshold"]).astype(int)
    acc = accuracy_score(y_test, preds)
    metrics = {
        "auc": float(auc),
        "accuracy_at_optimal_threshold": float(acc),
        "optimal_threshold": float(thr_info["threshold"]),
        "sensitivity_at_optimal_threshold": float(thr_info["tpr"]),
        "specificity_at_optimal_threshold": float(1.0 - thr_info["fpr"]),
    }
    print(f"\n[{model_name}]")
    print(json.dumps(metrics, indent=2))
    return metrics


def build_full_patient_time_scores_static(model, df, patient_ids, feature_cols):
    sub = df[df["patient_id"].isin(patient_ids)].copy().sort_values(["patient_id", "t"])
    sub["risk_score"] = model.predict_proba(sub[feature_cols])[:, 1]
    return sub


def build_full_patient_time_scores_rnn(model, df, patient_ids, feature_cols, imputer, scaler, seq_len=8):
    sub = df[df["patient_id"].isin(patient_ids)].copy().sort_values(["patient_id", "t"])
    rows = []
    for pid, g in tqdm(sub.groupby("patient_id"), desc="Building RNN scoring windows", total=sub["patient_id"].nunique()):
        g = g.sort_values("t").copy().reset_index(drop=True)
        if len(g) < seq_len:
            continue
        for end_t in range(seq_len - 1, len(g)):
            block = g.iloc[end_t - seq_len + 1:end_t + 1]
            rows.append({
                "patient_id": pid,
                "t": int(g.loc[end_t, "t"]),
                TARGET_COL: int(g.loc[end_t, TARGET_COL]),
                "seq": block[feature_cols].values,
            })

    if not rows:
        return pd.DataFrame(columns=["patient_id", "t", TARGET_COL, "risk_score"])

    eval_df = pd.DataFrame(rows)
    X = np.stack(eval_df["seq"].values)
    n, seq_len, n_feat = X.shape
    X2 = X.reshape(-1, n_feat)
    X2 = imputer.transform(X2)
    X2 = scaler.transform(X2)
    X = X2.reshape(n, seq_len, n_feat)
    eval_df["risk_score"] = predict_rnn(model, X)
    return eval_df[["patient_id", "t", TARGET_COL, "risk_score"]]


def compute_early_warning_metrics(score_df: pd.DataFrame, threshold: float, label_lead: int = 6) -> Dict[str, float]:
    ewt_list = []
    n_positive_patients = 0
    n_detected_positive_patients = 0
    false_alert_patients = 0
    n_negative_patients = 0

    for pid, g in score_df.groupby("patient_id"):
        g = g.sort_values("t").reset_index(drop=True).copy()
        pos = np.where(g[TARGET_COL].fillna(0).values.astype(int) == 1)[0]
        detected_times = g.loc[g["risk_score"] >= threshold, "t"].tolist()

        if len(pos) > 0:
            n_positive_patients += 1
            label_start_t = int(g.iloc[pos[0]]["t"])
            event_t = label_start_t + label_lead
            valid_detections = [t for t in detected_times if t < event_t]
            if valid_detections:
                det_t = int(valid_detections[0])
                ewt_list.append(event_t - det_t)
                n_detected_positive_patients += 1
        else:
            n_negative_patients += 1
            if len(detected_times) > 0:
                false_alert_patients += 1

    return {
        "n_positive_patients": int(n_positive_patients),
        "n_detected_positive_patients": int(n_detected_positive_patients),
        "patient_sensitivity": float(n_detected_positive_patients / n_positive_patients) if n_positive_patients else np.nan,
        "n_negative_patients": int(n_negative_patients),
        "false_alert_patient_rate": float(false_alert_patients / n_negative_patients) if n_negative_patients else np.nan,
        "median_early_warning_time_hours": float(np.median(ewt_list)) if ewt_list else np.nan,
        "mean_early_warning_time_hours": float(np.mean(ewt_list)) if ewt_list else np.nan,
    }


def evaluate_binary_subset(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    out = {
        "n_rows": int(len(y_true)),
        "n_positive_rows": int(np.sum(y_true == 1)),
        "n_negative_rows": int(np.sum(y_true == 0)),
        "auc": np.nan,
        "accuracy_at_optimal_threshold": np.nan,
        "optimal_threshold": np.nan,
        "sensitivity_at_optimal_threshold": np.nan,
        "specificity_at_optimal_threshold": np.nan,
        "brier_score": np.nan,
    }

    if len(y_true) == 0:
        return out

    out["brier_score"] = float(brier_score_loss(y_true, y_prob))
    if len(np.unique(y_true)) < 2:
        return out

    auc = roc_auc_score(y_true, y_prob)
    thr_info = optimal_threshold_from_roc(y_true, y_prob)
    preds = (y_prob >= thr_info["threshold"]).astype(int)
    acc = accuracy_score(y_true, preds)

    out.update({
        "auc": float(auc),
        "accuracy_at_optimal_threshold": float(acc),
        "optimal_threshold": float(thr_info["threshold"]),
        "sensitivity_at_optimal_threshold": float(thr_info["tpr"]),
        "specificity_at_optimal_threshold": float(1.0 - thr_info["fpr"]),
    })
    return out


def subgroup_row_metrics(eval_df: pd.DataFrame, group_col="group", target_col="proxy_label", prob_col="pred_prob") -> pd.DataFrame:
    rows = []
    for grp, g in eval_df.groupby(group_col):
        metrics = evaluate_binary_subset(g[target_col].values, g[prob_col].values)
        metrics[group_col] = grp
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)


def subgroup_patient_time_metrics(score_df: pd.DataFrame, threshold: float, label_lead: int = 6, group_col: str = "group") -> pd.DataFrame:
    rows = []
    for grp, g in score_df.groupby(group_col):
        metrics = compute_early_warning_metrics(
            g[["patient_id", "t", TARGET_COL, "risk_score"]].copy(),
            threshold=threshold,
            label_lead=label_lead,
        )
        metrics[group_col] = grp
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)


def calibration_table(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    df_cal = pd.DataFrame({"y_true": y_true, "y_prob": y_prob}).copy()
    if df_cal["y_prob"].nunique() < 2:
        return pd.DataFrame(columns=["bin", "mean_pred", "obs_rate", "n"])

    df_cal["bin"] = pd.qcut(df_cal["y_prob"], q=min(n_bins, df_cal["y_prob"].nunique()), duplicates="drop")
    out = (
        df_cal.groupby("bin")
        .agg(mean_pred=("y_prob", "mean"), obs_rate=("y_true", "mean"), n=("y_true", "size"))
        .reset_index()
    )
    return out


def threshold_metrics(y_true: np.ndarray, y_prob: np.ndarray, thresholds: List[float]) -> pd.DataFrame:
    rows = []
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        tp = int(np.sum((pred == 1) & (y_true == 1)))
        fp = int(np.sum((pred == 1) & (y_true == 0)))
        fn = int(np.sum((pred == 0) & (y_true == 1)))
        tn = int(np.sum((pred == 0) & (y_true == 0)))
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        ppv = tp / (tp + fp) if (tp + fp) else np.nan
        rows.append({
            "threshold": thr,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "sensitivity": sens,
            "specificity": spec,
            "false_positive_rate": fpr,
            "ppv": ppv,
        })
    return pd.DataFrame(rows)

## Data loading and preprocessing

In [4]:
# =========================
# Load data and build train/test sets
# =========================

set_global_seed(SEED)

files = find_patient_files(DATA_DIR, recursive=True)
print("Ready to load:", len(files), "files")

df, summaries = load_all_patients(DATA_DIR)
summary_preview = pd.DataFrame([s.__dict__ for s in summaries])

train_ids, test_ids = split_patients(summaries, test_size=TEST_SIZE, seed=SEED)

df_feat = add_missingness_and_deltas(df, ALL_FEATURES)
feature_cols_full = choose_feature_columns(df_feat, ALL_FEATURES)
feature_cols_physiology_only = choose_feature_columns(df_feat, PHYSIOLOGY_FEATURES)
feature_cols_no_demographics = choose_feature_columns(df_feat, [c for c in ALL_FEATURES if c not in ["Age", "Gender"]])
feature_cols_no_missingness = [c for c in feature_cols_full if not c.endswith("_missing")]
feature_cols_no_admin = choose_feature_columns(df_feat, [c for c in ALL_FEATURES if c not in ["Unit1", "Unit2", "HospAdmTime", "ICULOS"]])

train_rows = build_proxy_training_rows(df_feat, train_ids, event_window=LIU_LIKE_WINDOW, label_lead=SEPSIS_LABEL_LEAD)
test_rows = build_proxy_training_rows(df_feat, test_ids, event_window=LIU_LIKE_WINDOW, label_lead=SEPSIS_LABEL_LEAD)

X_train_full = train_rows[feature_cols_full]
y_train = train_rows["proxy_label"].astype(int).values
X_test_full = test_rows[feature_cols_full]
y_test = test_rows["proxy_label"].astype(int).values

print(f"Loaded {len(summaries)} patients")
print(f"Loaded {len(df):,} hourly rows")
print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Number of modeling features (full):", len(feature_cols_full))
print("Train shape:", X_train_full.shape, "| positives:", int(y_train.sum()), "| negatives:", int((y_train == 0).sum()))
print("Test shape:", X_test_full.shape, "| positives:", int(y_test.sum()), "| negatives:", int((y_test == 0).sum()))

Found 40336 patient files.
First few files:
 - data/training_setA/training/p000001.psv
 - data/training_setA/training/p000002.psv
 - data/training_setA/training/p000003.psv
 - data/training_setA/training/p000004.psv
 - data/training_setA/training/p000005.psv
Ready to load: 40336 files
Found 40336 patient files.
First few files:
 - data/training_setA/training/p000001.psv
 - data/training_setA/training/p000002.psv
 - data/training_setA/training/p000003.psv
 - data/training_setA/training/p000004.psv
 - data/training_setA/training/p000005.psv


Loading patient files:   0%|          | 0/40336 [00:00<?, ?it/s]

Adding missingness flags:   0%|          | 0/40 [00:00<?, ?it/s]

Computing deltas:   0%|          | 0/40336 [00:00<?, ?it/s]

Building proxy rows:   0%|          | 0/28235 [00:00<?, ?it/s]

Building proxy rows:   0%|          | 0/12101 [00:00<?, ?it/s]

Loaded 40336 patients
Loaded 1,552,210 hourly rows
Train patients: 28235
Test patients: 12101
Number of modeling features (full): 120
Train shape: (969736, 120) | positives: 4090 | negatives: 965646
Test shape: (415908, 120) | positives: 1754 | negatives: 414154


## Cohort overview and subgroup audit

In [5]:
# =========================
# Patient-level demographic table
# =========================

patient_demo = build_patient_demographics(df, summaries)
train_demo = patient_demo[patient_demo["patient_id"].isin(train_ids)].copy()
test_demo = patient_demo[patient_demo["patient_id"].isin(test_ids)].copy()

print("Summary preview:")
display(summary_preview.head())

print("Patient-level demographic preview:")
display(patient_demo.head())

cohort_summary = (
    patient_demo.groupby(["Gender_label", "Age_bin", "group"], dropna=False)
    .agg(
        n_patients=("patient_id", "nunique"),
        sepsis_prevalence=("became_septic", "mean"),
        mean_age=("Age", "mean"),
        median_age=("Age", "median"),
        mean_rows=("n_rows", "mean"),
        median_rows=("n_rows", "median"),
    )
    .reset_index()
    .sort_values(["Gender_label", "Age_bin"])
    .reset_index(drop=True)
)
cohort_summary["pct_patients"] = 100 * cohort_summary["n_patients"] / cohort_summary["n_patients"].sum()

print("Cohort summary by age × sex:")
display(cohort_summary)

Summary preview:


,patient_id,n_rows,onset_idx,became_septic
0,p000001,54,NaN,False
1,p000002,23,NaN,False
2,p000003,48,NaN,False
3,p000004,29,NaN,False
4,p000005,48,NaN,False


Patient-level demographic preview:


,patient_id,Age,Gender,SepsisLabel,ICULOS,became_septic,n_rows,Gender_label,Age_bin,group
0,p000001,83.14,0,0,1,False,54,Female,75+,Female | 75+
1,p000002,75.91,0,0,1,False,23,Female,75+,Female | 75+
2,p000003,45.82,0,0,1,False,48,Female,45-74,Female | 45-74
3,p000004,65.71,0,0,1,False,29,Female,45-74,Female | 45-74
4,p000005,28.09,1,0,2,False,48,Male,<45,Male | <45


Cohort summary by age × sex:


,Gender_label,Age_bin,group,n_patients,sepsis_prevalence,mean_age,median_age,mean_rows,median_rows,pct_patients
0,Female,45-74,Female | 45-74,10091,0.068279,61.270404,62.000,38.842335,39.0,25.017354
1,Female,75+,Female | 75+,4669,0.070036,82.211870,81.320,39.084601,39.0,11.575268
2,Female,<45,Female | <45,3010,0.058804,33.901150,35.025,36.432890,37.0,7.462317
3,Male,45-74,Male | 45-74,14462,0.074056,61.034971,61.440,38.431960,38.0,35.853828
4,Male,75+,Male | 75+,4726,0.082522,81.187499,80.180,39.346805,39.0,11.716581
5,Male,<45,Male | <45,3378,0.082297,34.310219,36.000,37.402901,37.0,8.374653


In [ ]:
# =========================
# Histogram of age by recorded gender
# =========================

plt.figure(figsize=(8, 5))
for gender in ["Female", "Male", "Missing/Other"]:
    sub = patient_demo.loc[patient_demo["Gender_label"] == gender, "Age"].dropna()
    if len(sub) > 0:
        plt.hist(sub, bins=30, alpha=0.5, label=gender)

plt.title("Distribution of patient ages by recorded gender")
plt.xlabel("Age")
plt.ylabel("Number of patients")
plt.legend()
plt.show()

In [7]:
# =========================
# Missingness audit by subgroup
# =========================

base_patient_rows = (
    df.sort_values(["patient_id", "t"])
    .groupby("patient_id", as_index=False)
    .first()[["patient_id"] + ALL_FEATURES]
    .copy()
)
base_patient_rows = base_patient_rows.merge(patient_demo[["patient_id", "group"]], on="patient_id", how="left")

missingness_by_group = []
for grp, g in base_patient_rows.groupby("group"):
    row = {"group": grp, "n_patients": g["patient_id"].nunique()}
    for col in ["HR", "O2Sat", "Temp", "SBP", "Resp", "WBC", "Creatinine", "Lactate"]:
        row[f"pct_missing_{col}"] = 100 * g[col].isna().mean()
    missingness_by_group.append(row)

missingness_by_group = pd.DataFrame(missingness_by_group).sort_values("group").reset_index(drop=True)
print("Selected baseline missingness rates by subgroup:")
display(missingness_by_group)

Selected baseline missingness rates by subgroup:


,group,n_patients,pct_missing_HR,pct_missing_O2Sat,pct_missing_Temp,pct_missing_SBP,pct_missing_Resp,pct_missing_WBC,pct_missing_Creatinine,pct_missing_Lactate
0,Female | 45-74,10091,0.009910,0.029729,0.564860,0.703597,0.188287,6.817957,5.519770,69.913785
1,Female | 75+,4669,0.021418,0.085671,0.663954,0.878132,0.192761,6.875134,5.525809,70.700364
2,Female | <45,3010,0.000000,0.033223,0.299003,0.265781,0.066445,8.205980,6.013289,76.744186
3,Male | 45-74,14462,0.006915,0.020744,0.822846,0.746785,0.186696,5.815240,4.356244,66.512239
4,Male | 75+,4726,0.000000,0.021160,1.015658,0.761744,0.105798,5.861193,4.909014,67.096911
5,Male | <45,3378,0.059207,0.177620,0.592066,0.532860,0.266430,7.430432,5.654233,70.663114


In [8]:
# =========================
# Measurement intensity audit by subgroup
# =========================

measurement_intensity = (
    df.groupby("patient_id")
    .agg(
        n_rows=("t", "size"),
        hr_density=("HR", lambda x: x.notna().mean()),
        lactate_density=("Lactate", lambda x: x.notna().mean()),
        wbc_density=("WBC", lambda x: x.notna().mean()),
        creatinine_density=("Creatinine", lambda x: x.notna().mean()),
    )
    .reset_index()
    .merge(patient_demo[["patient_id", "group"]], on="patient_id", how="left")
)

measurement_intensity_summary = (
    measurement_intensity.groupby("group")
    .agg(
        n_patients=("patient_id", "nunique"),
        mean_rows=("n_rows", "mean"),
        mean_hr_density=("hr_density", "mean"),
        mean_lactate_density=("lactate_density", "mean"),
        mean_wbc_density=("wbc_density", "mean"),
        mean_creatinine_density=("creatinine_density", "mean"),
    )
    .reset_index()
    .sort_values("group")
)

print("Measurement intensity by subgroup:")
display(measurement_intensity_summary)

Measurement intensity by subgroup:


,group,n_patients,mean_rows,mean_hr_density,mean_lactate_density,mean_wbc_density,mean_creatinine_density
0,Female | 45-74,10091,38.842335,0.882596,0.023277,0.063405,0.060637
1,Female | 75+,4669,39.084601,0.898325,0.024169,0.063469,0.059623
2,Female | <45,3010,36.432890,0.863162,0.018520,0.060912,0.064579
3,Male | 45-74,14462,38.431960,0.895661,0.028472,0.067586,0.062582
4,Male | 75+,4726,39.346805,0.905239,0.027418,0.066149,0.060594
5,Male | <45,3378,37.402901,0.877986,0.025661,0.063332,0.064398


## Train shared models and compute overall performance

In [9]:
# =========================
# Train shared GLM / XGBoost / GRU
# =========================

shared_models = {}
shared_row_metrics = {}
shared_patient_time_metrics = {}
shared_scores = {}

# GLM shared model
shared_models["GLM_full"] = train_glm(X_train_full, y_train)
shared_row_metrics["GLM_full"] = evaluate_static_model(shared_models["GLM_full"], X_test_full, y_test, "GLM_full")
joblib.dump(shared_models["GLM_full"], OUTPUT_DIR / "glm_full.joblib")

# Optional XGBoost shared model
if TRAIN_XGBOOST:
    shared_models["XGB_full"] = train_xgboost(X_train_full, y_train)
    shared_row_metrics["XGB_full"] = evaluate_static_model(shared_models["XGB_full"], X_test_full, y_test, "XGB_full")
    joblib.dump(shared_models["XGB_full"], OUTPUT_DIR / "xgb_full.joblib")
else:
    shared_models["XGB_full"] = None
    shared_row_metrics["XGB_full"] = None
    print("XGBoost training skipped")

# Optional GRU shared model
if TRAIN_GRU:
    train_seq_X, train_seq_y = build_patient_sequences(
        df_feat, train_ids, feature_cols_full,
        seq_len=SEQ_LEN,
        event_window=LIU_LIKE_WINDOW,
        label_lead=SEPSIS_LABEL_LEAD,
    )
    test_seq_X, test_seq_y = build_patient_sequences(
        df_feat, test_ids, feature_cols_full,
        seq_len=SEQ_LEN,
        event_window=LIU_LIKE_WINDOW,
        label_lead=SEPSIS_LABEL_LEAD,
    )

    train_seq_X_scaled, test_seq_X_scaled, rnn_imputer, rnn_scaler = median_impute_and_scale_for_rnn(train_seq_X, test_seq_X)
    print("Train sequences:", train_seq_X_scaled.shape)
    print("Test sequences:", test_seq_X_scaled.shape)

    idx = np.arange(len(train_seq_X_scaled))
    tr_idx, val_idx = train_test_split(
        idx,
        test_size=0.15,
        random_state=SEED,
        stratify=train_seq_y if len(set(train_seq_y)) > 1 else None,
    )

    shared_models["GRU_full"] = train_rnn(
        train_seq_X_scaled[tr_idx], train_seq_y[tr_idx],
        train_seq_X_scaled[val_idx], train_seq_y[val_idx],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
    )
    torch.save(shared_models["GRU_full"].state_dict(), OUTPUT_DIR / "gru_full.pt")

    rnn_probs = predict_rnn(shared_models["GRU_full"], test_seq_X_scaled)
    rnn_auc = roc_auc_score(test_seq_y, rnn_probs)
    rnn_thr = optimal_threshold_from_roc(test_seq_y, rnn_probs)
    rnn_preds = (rnn_probs >= rnn_thr["threshold"]).astype(int)
    rnn_acc = accuracy_score(test_seq_y, rnn_preds)
    shared_row_metrics["GRU_full"] = {
        "auc": float(rnn_auc),
        "accuracy_at_optimal_threshold": float(rnn_acc),
        "optimal_threshold": float(rnn_thr["threshold"]),
        "sensitivity_at_optimal_threshold": float(rnn_thr["tpr"]),
        "specificity_at_optimal_threshold": float(1.0 - rnn_thr["fpr"]),
    }
    print("\n[GRU_full]")
    print(json.dumps(shared_row_metrics["GRU_full"], indent=2))
else:
    shared_models["GRU_full"] = None
    shared_row_metrics["GRU_full"] = None
    test_seq_y = None
    rnn_imputer = None
    rnn_scaler = None
    rnn_probs = None
    print("GRU training skipped")


[GLM_full]
{
  "auc": 0.7872634455229306,
  "accuracy_at_optimal_threshold": 0.7606249459014974,
  "optimal_threshold": 0.48081935549965776,
  "sensitivity_at_optimal_threshold": 0.6887115165336374,
  "specificity_at_optimal_threshold": 0.7609295093129609
}
XGBoost training skipped


Building sequences:   0%|          | 0/28235 [00:00<?, ?it/s]

Building sequences:   0%|          | 0/12101 [00:00<?, ?it/s]

Train sequences: (785698, 8, 120)
Test sequences: (337047, 8, 120)


Training GRU:   0%|          | 0/12 [00:00<?, ?it/s]

...

Epoch 11: train_loss=0.1531, val_auc=0.9479


Epoch 12/12:   0%|          | 0/5218 [00:00<?, ?it/s]

Epoch 12: train_loss=0.1556, val_auc=0.9506


Predicting RNN probabilities:   0%|          | 0/1317 [00:00<?, ?it/s]


[GRU_full]
{
  "auc": 0.8016125456180063,
  "accuracy_at_optimal_threshold": 0.7551142718967978,
  "optimal_threshold": 0.00537517573684454,
  "sensitivity_at_optimal_threshold": 0.7020833333333333,
  "specificity_at_optimal_threshold": 0.7553418134901835
}


In [10]:
# =========================
# Shared-model patient-time scores and summary tables
# =========================

overall_results_rows = []

# GLM patient-time
shared_scores["GLM_full"] = build_full_patient_time_scores_static(shared_models["GLM_full"], df_feat, test_ids, feature_cols_full)
shared_patient_time_metrics["GLM_full"] = compute_early_warning_metrics(
    shared_scores["GLM_full"],
    shared_row_metrics["GLM_full"]["optimal_threshold"],
    label_lead=SEPSIS_LABEL_LEAD,
)
shared_scores["GLM_full"].to_csv(OUTPUT_DIR / "glm_full_test_time_scores.csv", index=False)

overall_results_rows.append({
    "model": "GLM_full",
    "row_auc": shared_row_metrics["GLM_full"]["auc"],
    "row_sensitivity": shared_row_metrics["GLM_full"]["sensitivity_at_optimal_threshold"],
    "row_specificity": shared_row_metrics["GLM_full"]["specificity_at_optimal_threshold"],
    "median_early_warning_hours": shared_patient_time_metrics["GLM_full"]["median_early_warning_time_hours"],
    "false_alert_patient_rate": shared_patient_time_metrics["GLM_full"]["false_alert_patient_rate"],
})

# XGB patient-time
if shared_models["XGB_full"] is not None:
    shared_scores["XGB_full"] = build_full_patient_time_scores_static(shared_models["XGB_full"], df_feat, test_ids, feature_cols_full)
    shared_patient_time_metrics["XGB_full"] = compute_early_warning_metrics(
        shared_scores["XGB_full"],
        shared_row_metrics["XGB_full"]["optimal_threshold"],
        label_lead=SEPSIS_LABEL_LEAD,
    )
    shared_scores["XGB_full"].to_csv(OUTPUT_DIR / "xgb_full_test_time_scores.csv", index=False)

    overall_results_rows.append({
        "model": "XGB_full",
        "row_auc": shared_row_metrics["XGB_full"]["auc"],
        "row_sensitivity": shared_row_metrics["XGB_full"]["sensitivity_at_optimal_threshold"],
        "row_specificity": shared_row_metrics["XGB_full"]["specificity_at_optimal_threshold"],
        "median_early_warning_hours": shared_patient_time_metrics["XGB_full"]["median_early_warning_time_hours"],
        "false_alert_patient_rate": shared_patient_time_metrics["XGB_full"]["false_alert_patient_rate"],
    })

# GRU patient-time
if shared_models["GRU_full"] is not None:
    shared_scores["GRU_full"] = build_full_patient_time_scores_rnn(
        shared_models["GRU_full"],
        df_feat,
        test_ids,
        feature_cols_full,
        rnn_imputer,
        rnn_scaler,
        seq_len=SEQ_LEN,
    )
    shared_patient_time_metrics["GRU_full"] = compute_early_warning_metrics(
        shared_scores["GRU_full"],
        shared_row_metrics["GRU_full"]["optimal_threshold"],
        label_lead=SEPSIS_LABEL_LEAD,
    )
    shared_scores["GRU_full"].to_csv(OUTPUT_DIR / "gru_full_test_time_scores.csv", index=False)

    overall_results_rows.append({
        "model": "GRU_full",
        "row_auc": shared_row_metrics["GRU_full"]["auc"],
        "row_sensitivity": shared_row_metrics["GRU_full"]["sensitivity_at_optimal_threshold"],
        "row_specificity": shared_row_metrics["GRU_full"]["specificity_at_optimal_threshold"],
        "median_early_warning_hours": shared_patient_time_metrics["GRU_full"]["median_early_warning_time_hours"],
        "false_alert_patient_rate": shared_patient_time_metrics["GRU_full"]["false_alert_patient_rate"],
    })

overall_results_df = pd.DataFrame(overall_results_rows)
print("Overall shared-model performance:")
display(overall_results_df)

Building RNN scoring windows:   0%|          | 0/12101 [00:00<?, ?it/s]

Predicting RNN probabilities:   0%|          | 0/1492 [00:00<?, ?it/s]

Overall shared-model performance:


,model,row_auc,row_sensitivity,row_specificity,median_early_warning_hours,false_alert_patient_rate
0,GLM_full,0.787263,0.688712,0.760930,34.0,0.642189
1,GRU_full,0.801613,0.702083,0.755342,33.0,0.820426


## Performance by age × sex subgroup

In [11]:
# =========================
# Build subgroup-ready evaluation frames for shared models
# =========================

test_rows_eval = test_rows.copy().merge(
    test_demo[["patient_id", "Gender_label", "Age_bin", "group"]],
    on="patient_id",
    how="left",
)

test_rows_eval["glm_pred_prob"] = shared_models["GLM_full"].predict_proba(X_test_full)[:, 1]

if shared_models["XGB_full"] is not None:
    test_rows_eval["xgb_pred_prob"] = shared_models["XGB_full"].predict_proba(X_test_full)[:, 1]

# GRU row-level subgroup frame
shared_gru_eval_rows = None
if shared_models["GRU_full"] is not None:
    shared_gru_eval_rows = []
    sub = df_feat[df_feat["patient_id"].isin(test_ids)].copy()

    for pid, g in tqdm(sub.groupby("patient_id"), desc="Building GRU subgroup eval rows", total=sub["patient_id"].nunique()):
        g = g.sort_values("t").reset_index(drop=True).copy()
        _, event_t = get_label_start_and_estimated_onset(g, TARGET_COL, label_lead=SEPSIS_LABEL_LEAD)

        if event_t is None:
            for end_t in range(SEQ_LEN - 1, len(g)):
                shared_gru_eval_rows.append({"patient_id": pid, "t": int(g.iloc[end_t]["t"]), "proxy_label": 0})
            continue

        start = event_t + LIU_LIKE_WINDOW[0]
        end = event_t + LIU_LIKE_WINDOW[1]
        for end_idx in range(len(g)):
            current_t = int(g.iloc[end_idx]["t"])
            if current_t < max(SEQ_LEN - 1, start) or current_t > end:
                continue
            block = g.iloc[end_idx - SEQ_LEN + 1:end_idx + 1]
            if len(block) == SEQ_LEN:
                shared_gru_eval_rows.append({"patient_id": pid, "t": int(g.iloc[end_idx]["t"]), "proxy_label": 1})

    shared_gru_eval_rows = pd.DataFrame(shared_gru_eval_rows)
    shared_gru_eval_rows["pred_prob"] = rnn_probs
    shared_gru_eval_rows = shared_gru_eval_rows.merge(
        test_demo[["patient_id", "Gender_label", "Age_bin", "group"]],
        on="patient_id",
        how="left",
    )

Building GRU subgroup eval rows:   0%|          | 0/12101 [00:00<?, ?it/s]

In [12]:
# =========================
# Subgroup row-level performance for shared models
# =========================

subgroup_row_results = []

glm_subgroup_row = subgroup_row_metrics(
    test_rows_eval.rename(columns={"glm_pred_prob": "pred_prob"}),
    group_col="group",
    target_col="proxy_label",
    prob_col="pred_prob",
)
glm_subgroup_row.insert(0, "model", "GLM_full")
subgroup_row_results.append(glm_subgroup_row)

if shared_models["XGB_full"] is not None:
    xgb_subgroup_row = subgroup_row_metrics(
        test_rows_eval.rename(columns={"xgb_pred_prob": "pred_prob"}),
        group_col="group",
        target_col="proxy_label",
        prob_col="pred_prob",
    )
    xgb_subgroup_row.insert(0, "model", "XGB_full")
    subgroup_row_results.append(xgb_subgroup_row)

if shared_models["GRU_full"] is not None and shared_gru_eval_rows is not None:
    gru_subgroup_row = subgroup_row_metrics(
        shared_gru_eval_rows,
        group_col="group",
        target_col="proxy_label",
        prob_col="pred_prob",
    )
    gru_subgroup_row.insert(0, "model", "GRU_full")
    subgroup_row_results.append(gru_subgroup_row)

subgroup_row_df = pd.concat(subgroup_row_results, ignore_index=True)
print("Shared-model row-level performance by age × sex subgroup:")
display(subgroup_row_df)

Shared-model row-level performance by age × sex subgroup:


,model,n_rows,n_positive_rows,n_negative_rows,auc,accuracy_at_optimal_threshold,optimal_threshold,sensitivity_at_optimal_threshold,specificity_at_optimal_threshold,brier_score,group
0,GLM_full,104081,458,103623,0.800836,0.791681,0.482823,0.700873,0.792083,0.159070,Female | 45-74
1,GLM_full,50324,216,50108,0.784147,0.833241,0.532313,0.662037,0.833979,0.164221,Female | 75+
2,GLM_full,31065,88,30977,0.786947,0.783390,0.475937,0.693182,0.783646,0.154244,Female | <45
3,GLM_full,147261,636,146625,0.783819,0.741581,0.472565,0.691824,0.741797,0.173092,Male | 45-74
4,GLM_full,49203,206,48997,0.770183,0.683149,0.459057,0.713592,0.683021,0.183564,Male | 75+
5,GLM_full,33974,150,33824,0.778448,0.720875,0.477513,0.740000,0.720790,0.186597,Male | <45
6,GRU_full,84360,372,83988,0.800828,0.777501,0.007460,0.674731,0.777956,0.037103,Female | 45-74
7,GRU_full,41004,185,40819,0.836056,0.811214,0.016697,0.718919,0.811632,0.033546,Female | 75+
8,GRU_full,24938,65,24873,0.896263,0.870358,0.041521,0.800000,0.870542,0.032881,Female | <45
9,GRU_full,119255,539,118716,0.800841,0.771926,0.005391,0.710575,0.772204,0.035012,Male | 45-74


In [13]:
# =========================
# Subgroup patient-time performance for shared models
# =========================

subgroup_pt_results = []

glm_scores_demo = shared_scores["GLM_full"].merge(
    test_demo[["patient_id", "Gender_label", "Age_bin", "group"]],
    on="patient_id",
    how="left",
)
glm_subgroup_pt = subgroup_patient_time_metrics(
    glm_scores_demo,
    threshold=shared_row_metrics["GLM_full"]["optimal_threshold"],
    label_lead=SEPSIS_LABEL_LEAD,
    group_col="group",
)
glm_subgroup_pt.insert(0, "model", "GLM_full")
subgroup_pt_results.append(glm_subgroup_pt)

if shared_models["XGB_full"] is not None:
    xgb_scores_demo = shared_scores["XGB_full"].merge(
        test_demo[["patient_id", "Gender_label", "Age_bin", "group"]],
        on="patient_id",
        how="left",
    )
    xgb_subgroup_pt = subgroup_patient_time_metrics(
        xgb_scores_demo,
        threshold=shared_row_metrics["XGB_full"]["optimal_threshold"],
        label_lead=SEPSIS_LABEL_LEAD,
        group_col="group",
    )
    xgb_subgroup_pt.insert(0, "model", "XGB_full")
    subgroup_pt_results.append(xgb_subgroup_pt)

if shared_models["GRU_full"] is not None:
    gru_scores_demo = shared_scores["GRU_full"].merge(
        test_demo[["patient_id", "Gender_label", "Age_bin", "group"]],
        on="patient_id",
        how="left",
    )
    gru_subgroup_pt = subgroup_patient_time_metrics(
        gru_scores_demo,
        threshold=shared_row_metrics["GRU_full"]["optimal_threshold"],
        label_lead=SEPSIS_LABEL_LEAD,
        group_col="group",
    )
    gru_subgroup_pt.insert(0, "model", "GRU_full")
    subgroup_pt_results.append(gru_subgroup_pt)

subgroup_patient_time_df = pd.concat(subgroup_pt_results, ignore_index=True)
print("Shared-model patient-time performance by age × sex subgroup:")
display(subgroup_patient_time_df)

Shared-model patient-time performance by age × sex subgroup:


,model,n_positive_patients,n_detected_positive_patients,patient_sensitivity,n_negative_patients,false_alert_patient_rate,median_early_warning_time_hours,mean_early_warning_time_hours,group
0,GLM_full,230,202,0.878261,2805,0.590018,38.0,60.722772,Female | 45-74
1,GLM_full,108,96,0.888889,1327,0.641296,48.5,58.114583,Female | 75+
2,GLM_full,44,36,0.818182,872,0.535550,46.0,54.611111,Female | <45
3,GLM_full,318,279,0.877358,3987,0.664409,31.0,52.164875,Male | 45-74
4,GLM_full,104,92,0.884615,1295,0.749807,29.0,55.684783,Male | 75+
5,GLM_full,76,67,0.881579,935,0.655615,24.0,37.343284,Male | <45
6,GRU_full,230,198,0.860870,2805,0.813547,37.0,60.202020,Female | 45-74
7,GRU_full,108,95,0.879630,1327,0.855313,39.0,57.221053,Female | 75+
8,GRU_full,44,35,0.795455,872,0.815367,40.0,52.085714,Female | <45
9,GRU_full,318,272,0.855346,3987,0.802358,32.0,51.507353,Male | 45-74


In [14]:
# =========================
# Compact overall + subgroup table
# =========================

overall_row_results = []
for model_name, metrics in shared_row_metrics.items():
    if metrics is None:
        continue
    n_rows = len(y_test) if model_name != "GRU_full" else len(test_seq_y)
    n_positive = int(np.sum(y_test == 1)) if model_name != "GRU_full" else int(np.sum(test_seq_y == 1))
    n_negative = int(np.sum(y_test == 0)) if model_name != "GRU_full" else int(np.sum(test_seq_y == 0))
    overall_row_results.append({
        "model": model_name,
        "group": "Overall",
        "n_rows": n_rows,
        "n_positive_rows": n_positive,
        "n_negative_rows": n_negative,
        **metrics,
    })

overall_row_df = pd.DataFrame(overall_row_results)
row_comparison_table = pd.concat([overall_row_df, subgroup_row_df], ignore_index=True)
row_comparison_table = row_comparison_table.sort_values(["model", "group"]).reset_index(drop=True)
print("Overall + subgroup row-level comparison table:")
display(row_comparison_table)

Overall + subgroup row-level comparison table:


,model,group,n_rows,n_positive_rows,n_negative_rows,auc,accuracy_at_optimal_threshold,optimal_threshold,sensitivity_at_optimal_threshold,specificity_at_optimal_threshold,brier_score
0,GLM_full,Female | 45-74,104081,458,103623,0.800836,0.791681,0.482823,0.700873,0.792083,0.159070
1,GLM_full,Female | 75+,50324,216,50108,0.784147,0.833241,0.532313,0.662037,0.833979,0.164221
2,GLM_full,Female | <45,31065,88,30977,0.786947,0.783390,0.475937,0.693182,0.783646,0.154244
3,GLM_full,Male | 45-74,147261,636,146625,0.783819,0.741581,0.472565,0.691824,0.741797,0.173092
4,GLM_full,Male | 75+,49203,206,48997,0.770183,0.683149,0.459057,0.713592,0.683021,0.183564
5,GLM_full,Male | <45,33974,150,33824,0.778448,0.720875,0.477513,0.740000,0.720790,0.186597
6,GLM_full,Overall,415908,1754,414154,0.787263,0.760625,0.480819,0.688712,0.760930,NaN
7,GRU_full,Female | 45-74,84360,372,83988,0.800828,0.777501,0.007460,0.674731,0.777956,0.037103
8,GRU_full,Female | 75+,41004,185,40819,0.836056,0.811214,0.016697,0.718919,0.811632,0.033546
9,GRU_full,Female | <45,24938,65,24873,0.896263,0.870358,0.041521,0.800000,0.870542,0.032881


## Calibration and threshold tradeoff analysis

In [15]:
# =========================
# Calibration tables for shared GLM by subgroup
# =========================

shared_glm_calibration = {}
for grp, g in test_rows_eval.groupby("group"):
    shared_glm_calibration[grp] = calibration_table(
        g["proxy_label"].values,
        g["glm_pred_prob"].values,
        n_bins=CALIBRATION_BINS,
    )

example_group = sorted(shared_glm_calibration.keys())[0]
print(f"Example GLM calibration table for group: {example_group}")
display(shared_glm_calibration[example_group])

Example GLM calibration table for group: Female | 45-74


,bin,mean_pred,obs_rate,n
0,"(-0.0009989, 0.148]",0.082664,0.000865,10409
1,"(0.148, 0.202]",0.176779,0.000865,10408
2,"(0.202, 0.246]",0.225136,0.001729,10408
3,"(0.246, 0.286]",0.266220,0.001633,10408
4,"(0.286, 0.326]",0.305943,0.001345,10408
5,"(0.326, 0.37]",0.347522,0.002498,10408
6,"(0.37, 0.423]",0.395645,0.001922,10408
7,"(0.423, 0.491]",0.455160,0.002978,10408
8,"(0.491, 0.605]",0.541469,0.006245,10408
9,"(0.605, 1.0]",0.744268,0.023924,10408


In [ ]:
# =========================
# Calibration plot for shared GLM by subgroup
# =========================

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle="--")
for grp, cal_df in shared_glm_calibration.items():
    if len(cal_df) > 0:
        plt.plot(cal_df["mean_pred"], cal_df["obs_rate"], marker="o", label=grp)

plt.xlabel("Mean predicted risk")
plt.ylabel("Observed event rate")
plt.title("GLM calibration by age × sex subgroup")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [17]:
# =========================
# Threshold tradeoff table for shared GLM by subgroup
# =========================

threshold_grid = np.round(np.linspace(0.1, 0.9, 9), 2)
glm_threshold_tradeoff = []
for grp, g in test_rows_eval.groupby("group"):
    trade = threshold_metrics(g["proxy_label"].values, g["glm_pred_prob"].values, threshold_grid)
    trade.insert(0, "group", grp)
    glm_threshold_tradeoff.append(trade)

glm_threshold_tradeoff_df = pd.concat(glm_threshold_tradeoff, ignore_index=True)
print("Threshold tradeoff table for shared GLM:")
display(glm_threshold_tradeoff_df.head(20))

Threshold tradeoff table for shared GLM:


,group,threshold,tp,fp,fn,tn,sensitivity,specificity,false_positive_rate,ppv
0,Female | 45-74,0.1,456,98660,2,4963,0.995633,0.047895,0.952105,0.004601
1,Female | 45-74,0.2,441,83371,17,20252,0.962882,0.195439,0.804561,0.005262
2,Female | 45-74,0.3,401,58430,57,45193,0.875546,0.436129,0.563871,0.006816
3,Female | 45-74,0.4,353,35115,105,68508,0.770742,0.661127,0.338873,0.009953
4,Female | 45-74,0.5,308,19412,150,84211,0.672489,0.812667,0.187333,0.015619
5,Female | 45-74,0.6,251,10498,207,93125,0.548035,0.898690,0.101310,0.023351
6,Female | 45-74,0.7,193,5605,265,98018,0.421397,0.945910,0.054090,0.033287
7,Female | 45-74,0.8,151,2827,307,100796,0.329694,0.972718,0.027282,0.050705
8,Female | 45-74,0.9,99,1156,359,102467,0.216157,0.988844,0.011156,0.078884
9,Female | 75+,0.1,216,48042,0,2066,1.000000,0.041231,0.958769,0.004476


## Process-bias and ablation analysis

This section compares models that intentionally remove some information sources to see whether subgroup disparities are driven by:

- explicit demographic covariates (`Age`, `Gender`)
- missingness indicators
- administrative / site-process variables
- non-physiology variables more broadly

In [18]:
# =========================
# Train ablation GLMs
# =========================

ablation_feature_sets = {
    "GLM_full": feature_cols_full,
    "GLM_physiology_only": feature_cols_physiology_only,
    "GLM_no_age_gender": feature_cols_no_demographics,
    "GLM_no_missingness": feature_cols_no_missingness,
    "GLM_no_admin": feature_cols_no_admin,
}

ablation_models = {}
ablation_overall = []
ablation_subgroups = []

for model_name, cols in ablation_feature_sets.items():
    print(f"Training {model_name} with {len(cols)} features")
    Xtr = train_rows[cols]
    Xte = test_rows[cols]
    model = train_glm(Xtr, y_train)
    metrics = evaluate_static_model(model, Xte, y_test, model_name)

    tmp_eval = test_rows_eval[["patient_id", "proxy_label", "Gender_label", "Age_bin", "group"]].copy()
    tmp_eval["pred_prob"] = model.predict_proba(Xte)[:, 1]
    subgroup_metrics = subgroup_row_metrics(tmp_eval, group_col="group", target_col="proxy_label", prob_col="pred_prob")
    subgroup_metrics.insert(0, "model", model_name)

    ablation_models[model_name] = model
    ablation_overall.append({"model": model_name, **metrics, "n_features": len(cols)})
    ablation_subgroups.append(subgroup_metrics)

ablation_overall_df = pd.DataFrame(ablation_overall).sort_values("model").reset_index(drop=True)
ablation_subgroup_df = pd.concat(ablation_subgroups, ignore_index=True)

print("Overall ablation performance:")
display(ablation_overall_df)

print("Subgroup ablation performance:")
display(ablation_subgroup_df)

Training GLM_full with 120 features

[GLM_full]
{
  "auc": 0.7872634455229306,
  "accuracy_at_optimal_threshold": 0.7606249459014974,
  "optimal_threshold": 0.48081935549965776,
  "sensitivity_at_optimal_threshold": 0.6887115165336374,
  "specificity_at_optimal_threshold": 0.7609295093129609
}
Training GLM_physiology_only with 102 features

[GLM_physiology_only]
{
  "auc": 0.7015922380466839,
  "accuracy_at_optimal_threshold": 0.7063124537157256,
  "optimal_threshold": 0.4880103560037626,
  "sensitivity_at_optimal_threshold": 0.5986316989737742,
  "specificity_at_optimal_threshold": 0.7067684967427575
}
Training GLM_no_age_gender with 114 features

[GLM_no_age_gender]
{
  "auc": 0.7854438083005265,
  "accuracy_at_optimal_threshold": 0.7494830587533782,
  "optimal_threshold": 0.4729578302537605,
  "sensitivity_at_optimal_threshold": 0.6887115165336374,
  "specificity_at_optimal_threshold": 0.7497404347175205
}
Training GLM_no_missingness with 80 features

[GLM_no_missingness]
{
  "auc":

,model,auc,accuracy_at_optimal_threshold,optimal_threshold,sensitivity_at_optimal_threshold,specificity_at_optimal_threshold,n_features
0,GLM_full,0.787263,0.760625,0.480819,0.688712,0.760930,120
1,GLM_no_admin,0.702365,0.648833,0.458725,0.647092,0.648841,108
2,GLM_no_age_gender,0.785444,0.749483,0.472958,0.688712,0.749740,114
3,GLM_no_missingness,0.743786,0.716086,0.478185,0.655074,0.716345,80
4,GLM_physiology_only,0.701592,0.706312,0.488010,0.598632,0.706768,102


Subgroup ablation performance:


,model,n_rows,n_positive_rows,n_negative_rows,auc,accuracy_at_optimal_threshold,optimal_threshold,sensitivity_at_optimal_threshold,specificity_at_optimal_threshold,brier_score,group
0,GLM_full,104081,458,103623,0.800836,0.791681,0.482823,0.700873,0.792083,0.159070,Female | 45-74
1,GLM_full,50324,216,50108,0.784147,0.833241,0.532313,0.662037,0.833979,0.164221,Female | 75+
2,GLM_full,31065,88,30977,0.786947,0.783390,0.475937,0.693182,0.783646,0.154244,Female | <45
3,GLM_full,147261,636,146625,0.783819,0.741581,0.472565,0.691824,0.741797,0.173092,Male | 45-74
4,GLM_full,49203,206,48997,0.770183,0.683149,0.459057,0.713592,0.683021,0.183564,Male | 75+
5,GLM_full,33974,150,33824,0.778448,0.720875,0.477513,0.740000,0.720790,0.186597,Male | <45
6,GLM_physiology_only,104081,458,103623,0.720825,0.620670,0.448468,0.716157,0.620248,0.211705,Female | 45-74
7,GLM_physiology_only,50324,216,50108,0.690568,0.706462,0.494242,0.601852,0.706913,0.212976,Female | 75+
8,GLM_physiology_only,31065,88,30977,0.684009,0.691582,0.488232,0.625000,0.691771,0.214660,Female | <45
9,GLM_physiology_only,147261,636,146625,0.698047,0.651415,0.452270,0.639937,0.651465,0.204704,Male | 45-74


In [19]:
# =========================
# Disparity summary across ablations
# =========================

def disparity_summary(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    for model_name, g in df.groupby("model"):
        vals = g[metric].dropna().values
        rows.append({
            "model": model_name,
            f"min_{metric}": np.min(vals) if len(vals) else np.nan,
            f"max_{metric}": np.max(vals) if len(vals) else np.nan,
            f"gap_{metric}": (np.max(vals) - np.min(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)

auc_gap_df = disparity_summary(ablation_subgroup_df, "auc")
sens_gap_df = disparity_summary(ablation_subgroup_df, "sensitivity_at_optimal_threshold")
spec_gap_df = disparity_summary(ablation_subgroup_df, "specificity_at_optimal_threshold")

disparity_gap_df = auc_gap_df.merge(sens_gap_df, on="model", how="outer").merge(spec_gap_df, on="model", how="outer")
print("Disparity-gap summary across GLM ablations:")
display(disparity_gap_df)

Disparity-gap summary across GLM ablations:


,model,min_auc,max_auc,gap_auc,min_sensitivity_at_optimal_threshold,max_sensitivity_at_optimal_threshold,gap_sensitivity_at_optimal_threshold,min_specificity_at_optimal_threshold,max_specificity_at_optimal_threshold,gap_specificity_at_optimal_threshold
0,GLM_full,0.770183,0.800836,0.030653,0.662037,0.740000,0.077963,0.683021,0.833979,0.150957
1,GLM_no_admin,0.683854,0.721195,0.037342,0.580000,0.731441,0.151441,0.609942,0.771641,0.161700
2,GLM_no_age_gender,0.769569,0.800935,0.031366,0.662037,0.733333,0.071296,0.700390,0.835914,0.135525
3,GLM_no_missingness,0.703755,0.761916,0.058161,0.606796,0.673333,0.066537,0.663759,0.810148,0.146389
4,GLM_physiology_only,0.683958,0.720825,0.036867,0.580000,0.716157,0.136157,0.620248,0.772085,0.151837


## Optional subgroup-specific model training

By default this notebook trains **shared models** and evaluates them in each subgroup.

If you want to directly compare a shared model to **separate age-sex-specific models**, set:

```python
TRAIN_GROUP_SPECIFIC_GLM = True
```

near the top of the notebook.

In [20]:
# =========================
# Optional subgroup-specific GLMs
# =========================

group_specific_results = []
group_specific_models = {}

if TRAIN_GROUP_SPECIFIC_GLM:
    train_demo_map = train_demo[["patient_id", "group"]].copy()
    train_rows_grouped = train_rows.merge(train_demo_map, on="patient_id", how="left")
    test_rows_grouped = test_rows.merge(test_demo[["patient_id", "group"]], on="patient_id", how="left")

    group_counts = train_demo.groupby("group")["patient_id"].nunique().reset_index(name="n_train_patients")
    eligible_groups = group_counts.loc[
        group_counts["n_train_patients"] >= MIN_GROUP_PATIENTS_FOR_SUBGROUP_MODEL,
        "group"
    ].tolist()

    print("Eligible groups for subgroup-specific GLMs:", eligible_groups)

    for grp in eligible_groups:
        tr = train_rows_grouped[train_rows_grouped["group"] == grp].copy()
        te = test_rows_grouped[test_rows_grouped["group"] == grp].copy()

        if len(tr) == 0 or len(te) == 0:
            continue
        if tr["proxy_label"].nunique() < 2 or te["proxy_label"].nunique() < 2:
            continue

        model = train_glm(tr[feature_cols_full], tr["proxy_label"].astype(int).values)
        probs = model.predict_proba(te[feature_cols_full])[:, 1]
        metrics = evaluate_binary_subset(te["proxy_label"].astype(int).values, probs)

        shared_metrics_for_group = subgroup_row_df[
            (subgroup_row_df["model"] == "GLM_full") & (subgroup_row_df["group"] == grp)
        ]
        shared_auc = shared_metrics_for_group["auc"].iloc[0] if len(shared_metrics_for_group) else np.nan

        group_specific_results.append({
            "group": grp,
            "n_train_rows": len(tr),
            "n_test_rows": len(te),
            "shared_glm_auc": shared_auc,
            "group_specific_glm_auc": metrics["auc"],
            "shared_minus_group_specific_auc": shared_auc - metrics["auc"] if pd.notna(shared_auc) and pd.notna(metrics["auc"]) else np.nan,
            "group_specific_sensitivity": metrics["sensitivity_at_optimal_threshold"],
            "group_specific_specificity": metrics["specificity_at_optimal_threshold"],
        })
        group_specific_models[grp] = model

    group_specific_results_df = pd.DataFrame(group_specific_results).sort_values("group").reset_index(drop=True)
    print("Shared vs subgroup-specific GLM comparison:")
    display(group_specific_results_df)
else:
    group_specific_results_df = pd.DataFrame()
    print("TRAIN_GROUP_SPECIFIC_GLM is False; subgroup-specific model training skipped.")

TRAIN_GROUP_SPECIFIC_GLM is False; subgroup-specific model training skipped.


## Save outputs

In [21]:
# =========================
# Save outputs
# =========================

cohort_summary.to_csv(OUTPUT_DIR / "cohort_summary_by_group.csv", index=False)
missingness_by_group.to_csv(OUTPUT_DIR / "missingness_by_group.csv", index=False)
measurement_intensity_summary.to_csv(OUTPUT_DIR / "measurement_intensity_by_group.csv", index=False)
overall_results_df.to_csv(OUTPUT_DIR / "overall_shared_model_results.csv", index=False)
subgroup_row_df.to_csv(OUTPUT_DIR / "shared_model_subgroup_row_metrics.csv", index=False)
subgroup_patient_time_df.to_csv(OUTPUT_DIR / "shared_model_subgroup_patient_time_metrics.csv", index=False)
row_comparison_table.to_csv(OUTPUT_DIR / "overall_plus_subgroup_row_comparison.csv", index=False)
glm_threshold_tradeoff_df.to_csv(OUTPUT_DIR / "glm_threshold_tradeoff_by_group.csv", index=False)
ablation_overall_df.to_csv(OUTPUT_DIR / "ablation_overall_results.csv", index=False)
ablation_subgroup_df.to_csv(OUTPUT_DIR / "ablation_subgroup_results.csv", index=False)
disparity_gap_df.to_csv(OUTPUT_DIR / "ablation_disparity_gaps.csv", index=False)

for grp, cal_df in shared_glm_calibration.items():
    safe_name = grp.replace(" ", "_").replace("|", "-")
    cal_df.to_csv(OUTPUT_DIR / f"calibration_{safe_name}.csv", index=False)

if len(group_specific_results_df) > 0:
    group_specific_results_df.to_csv(OUTPUT_DIR / "group_specific_glm_comparison.csv", index=False)

with open(OUTPUT_DIR / "notebook_run_config.json", "w") as f:
    json.dump({
        "SEED": SEED,
        "TEST_SIZE": TEST_SIZE,
        "LIU_LIKE_WINDOW": LIU_LIKE_WINDOW,
        "SEPSIS_LABEL_LEAD": SEPSIS_LABEL_LEAD,
        "AGE_BIN_LABELS": AGE_BIN_LABELS,
        "TRAIN_XGBOOST": TRAIN_XGBOOST,
        "TRAIN_GRU": TRAIN_GRU,
        "TRAIN_GROUP_SPECIFIC_GLM": TRAIN_GROUP_SPECIFIC_GLM,
    }, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())

Saved outputs to: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/results_fairness_audit


In [22]:
# =========================
# Statistical tests for subgroup disparities
# =========================

from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu
from sklearn.utils import resample

def bootstrap_auc(y_true, y_prob, n_boot=200):
    aucs = []
    for _ in range(n_boot):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.array(aucs)


def compare_groups_auc(eval_df, group_a, group_b):
    g1 = eval_df[eval_df["group"] == group_a]
    g2 = eval_df[eval_df["group"] == group_b]

    auc1 = roc_auc_score(g1["proxy_label"], g1["glm_pred_prob"])
    auc2 = roc_auc_score(g2["proxy_label"], g2["glm_pred_prob"])

    boot1 = bootstrap_auc(g1["proxy_label"].values, g1["glm_pred_prob"].values)
    boot2 = bootstrap_auc(g2["proxy_label"].values, g2["glm_pred_prob"].values)

    stat, pval = mannwhitneyu(boot1, boot2, alternative="two-sided")

    return {
        "group_a": group_a,
        "group_b": group_b,
        "auc_a": auc1,
        "auc_b": auc2,
        "auc_diff": auc1 - auc2,
        "p_value": pval
    }


def compare_groups_error_rates(eval_df, threshold, group_a, group_b):
    def confusion(g):
        pred = (g["glm_pred_prob"] >= threshold).astype(int)
        y = g["proxy_label"]

        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        fn = np.sum((pred == 0) & (y == 1))
        tn = np.sum((pred == 0) & (y == 0))

        return tp, fp, fn, tn

    g1 = eval_df[eval_df["group"] == group_a]
    g2 = eval_df[eval_df["group"] == group_b]

    tp1, fp1, fn1, tn1 = confusion(g1)
    tp2, fp2, fn2, tn2 = confusion(g2)

    # Compare false positive rates
    table_fp = np.array([
        [fp1, tn1],
        [fp2, tn2]
    ])
    _, p_fp, _, _ = chi2_contingency(table_fp)

    # Compare false negative rates
    table_fn = np.array([
        [fn1, tp1],
        [fn2, tp2]
    ])
    _, p_fn, _, _ = chi2_contingency(table_fn)

    return {
        "group_a": group_a,
        "group_b": group_b,
        "fp_rate_a": fp1 / (fp1 + tn1),
        "fp_rate_b": fp2 / (fp2 + tn2),
        "fn_rate_a": fn1 / (fn1 + tp1),
        "fn_rate_b": fn2 / (fn2 + tp2),
        "p_value_fp_rate": p_fp,
        "p_value_fn_rate": p_fn
    }

In [23]:
# =========================
# Pairwise subgroup comparisons
# =========================

groups = sorted(test_rows_eval["group"].dropna().unique())
threshold = shared_row_metrics["GLM_full"]["optimal_threshold"]

auc_tests = []
error_tests = []

for i in range(len(groups)):
    for j in range(i + 1, len(groups)):
        g1, g2 = groups[i], groups[j]

        auc_tests.append(compare_groups_auc(test_rows_eval, g1, g2))
        error_tests.append(compare_groups_error_rates(test_rows_eval, threshold, g1, g2))

auc_tests_df = pd.DataFrame(auc_tests)
error_tests_df = pd.DataFrame(error_tests)

print("AUC difference tests:")
display(auc_tests_df.sort_values("p_value"))

print("Error rate difference tests:")
display(error_tests_df.sort_values("p_value_fp_rate"))

AUC difference tests:


,group_a,group_b,auc_a,auc_b,auc_diff,p_value
3,Female | 45-74,Male | 75+,0.800836,0.770183,0.030653,2.591300e-49
2,Female | 45-74,Male | 45-74,0.800836,0.783819,0.017017,6.503777e-43
4,Female | 45-74,Male | <45,0.800836,0.778448,0.022388,1.249249e-34
0,Female | 45-74,Female | 75+,0.800836,0.784147,0.016690,8.833166e-23
12,Male | 45-74,Male | 75+,0.783819,0.770183,0.013636,5.696621e-21
10,Female | <45,Male | 75+,0.786947,0.770183,0.016763,8.960066e-14
7,Female | 75+,Male | 75+,0.784147,0.770183,0.013963,5.629626e-11
1,Female | 45-74,Female | <45,0.800836,0.786947,0.013889,1.396246e-09
14,Male | 75+,Male | <45,0.770183,0.778448,-0.008265,2.232488e-07
11,Female | <45,Male | <45,0.786947,0.778448,0.008498,1.372292e-04


Error rate difference tests:


,group_a,group_b,fp_rate_a,fp_rate_b,fn_rate_a,fn_rate_b,p_value_fp_rate,p_value_fn_rate
3,Female | 45-74,Male | 75+,0.210542,0.279527,0.299127,0.330097,9.244240e-195,0.478742
4,Female | 45-74,Male | <45,0.210542,0.274450,0.299127,0.266667,2.589232e-131,0.511824
10,Female | <45,Male | 75+,0.210834,0.279527,0.318182,0.330097,5.288112e-105,0.949181
2,Female | 45-74,Male | 45-74,0.210542,0.246916,0.299127,0.322327,4.766451e-100,0.452916
11,Female | <45,Male | <45,0.210834,0.274450,0.318182,0.266667,4.775557e-79,0.483537
7,Female | 75+,Male | 75+,0.229125,0.279527,0.314815,0.330097,3.846145e-74,0.816858
8,Female | 75+,Male | <45,0.229125,0.274450,0.314815,0.266667,2.496835e-50,0.380618
12,Male | 45-74,Male | 75+,0.246916,0.279527,0.322327,0.330097,1.400494e-46,0.903349
9,Female | <45,Male | 45-74,0.210834,0.246916,0.318182,0.322327,1.758665e-41,1.000000
13,Male | 45-74,Male | <45,0.246916,0.274450,0.322327,0.266667,8.112162e-26,0.220236


In [24]:
# =========================
# Learn group-specific thresholds (equalize FPR)
# =========================

def find_threshold_for_target_fpr(y_true, y_prob, target_fpr):
    thresholds = np.linspace(0.01, 0.99, 200)
    best_thr = 0.5
    best_diff = np.inf

    for thr in thresholds:
        preds = (y_prob >= thr).astype(int)
        fp = np.sum((preds == 1) & (y_true == 0))
        tn = np.sum((preds == 0) & (y_true == 0))

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        diff = abs(fpr - target_fpr)

        if diff < best_diff:
            best_diff = diff
            best_thr = thr

    return best_thr


# Use GLM as baseline model
baseline_probs = test_rows_eval["glm_pred_prob"].values
baseline_y = test_rows_eval["proxy_label"].values

# Global threshold (from earlier)
global_threshold = shared_row_metrics["GLM_full"]["optimal_threshold"]

# Compute baseline FPR
baseline_preds = (baseline_probs >= global_threshold).astype(int)
fp = np.sum((baseline_preds == 1) & (baseline_y == 0))
tn = np.sum((baseline_preds == 0) & (baseline_y == 0))
target_fpr = fp / (fp + tn)

print("Target (global) FPR:", target_fpr)

# Learn per-group thresholds
group_thresholds = {}

for grp, g in test_rows_eval.groupby("group"):
    thr = find_threshold_for_target_fpr(
        g["proxy_label"].values,
        g["glm_pred_prob"].values,
        target_fpr
    )
    group_thresholds[grp] = thr

print("Learned group-specific thresholds:")
for k, v in group_thresholds.items():
    print(f"{k}: {v:.4f}")

Target (global) FPR: 0.23907049068703912
Learned group-specific thresholds:
Female | 45-74: 0.4581
Female | 75+: 0.4729
Female | <45: 0.4581
Male | 45-74: 0.4877
Male | 75+: 0.5025
Male | <45: 0.5074


In [25]:
# =========================
# Apply group-specific thresholds
# =========================

def apply_group_thresholds(df, prob_col, group_col, thresholds):
    preds = []
    for _, row in df.iterrows():
        grp = row[group_col]
        thr = thresholds.get(grp, 0.5)
        preds.append(int(row[prob_col] >= thr))
    return np.array(preds)


test_rows_eval["pred_global"] = (test_rows_eval["glm_pred_prob"] >= global_threshold).astype(int)
test_rows_eval["pred_group_adjusted"] = apply_group_thresholds(
    test_rows_eval,
    prob_col="glm_pred_prob",
    group_col="group",
    thresholds=group_thresholds
)

In [26]:
# =========================
# Compare fairness (global vs group thresholds)
# =========================

def compute_group_error_rates(df, pred_col):
    rows = []
    for grp, g in df.groupby("group"):
        y = g["proxy_label"].values
        pred = g[pred_col].values

        fp = np.sum((pred == 1) & (y == 0))
        tn = np.sum((pred == 0) & (y == 0))
        fn = np.sum((pred == 0) & (y == 1))
        tp = np.sum((pred == 1) & (y == 1))

        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        fnr = fn / (fn + tp) if (fn + tp) else np.nan

        rows.append({
            "group": grp,
            "FPR": fpr,
            "FNR": fnr,
        })

    return pd.DataFrame(rows)


baseline_fairness = compute_group_error_rates(test_rows_eval, "pred_global")
adjusted_fairness = compute_group_error_rates(test_rows_eval, "pred_group_adjusted")

comparison = baseline_fairness.merge(
    adjusted_fairness,
    on="group",
    suffixes=("_global", "_adjusted")
)

print("Fairness comparison:")
display(comparison)

Fairness comparison:


,group,FPR_global,FNR_global,FPR_adjusted,FNR_adjusted
0,Female | 45-74,0.210542,0.299127,0.242427,0.281659
1,Female | 75+,0.229125,0.314815,0.239922,0.314815
2,Female | <45,0.210834,0.318182,0.238500,0.306818
3,Male | 45-74,0.246916,0.322327,0.237033,0.336478
4,Male | 75+,0.279527,0.330097,0.241852,0.378641
5,Male | <45,0.274450,0.266667,0.236341,0.353333


In [27]:
# =========================
# Disparity reduction summary
# =========================

def disparity_gap(df, col):
    return df[col].max() - df[col].min()

summary = {
    "FPR_gap_global": disparity_gap(comparison, "FPR_global"),
    "FPR_gap_adjusted": disparity_gap(comparison, "FPR_adjusted"),
    "FNR_gap_global": disparity_gap(comparison, "FNR_global"),
    "FNR_gap_adjusted": disparity_gap(comparison, "FNR_adjusted"),
}

summary_df = pd.DataFrame([summary])

print("Disparity gap reduction:")
display(summary_df)

Disparity gap reduction:


,FPR_gap_global,FPR_gap_adjusted,FNR_gap_global,FNR_gap_adjusted
0,0.068985,0.006086,0.06343,0.096981


In [28]:
# =========================
# Performance impact (AUC + accuracy)
# =========================

from sklearn.metrics import accuracy_score

# AUC does NOT change (same probabilities)
auc = roc_auc_score(test_rows_eval["proxy_label"], test_rows_eval["glm_pred_prob"])

acc_global = accuracy_score(
    test_rows_eval["proxy_label"],
    test_rows_eval["pred_global"]
)

acc_adjusted = accuracy_score(
    test_rows_eval["proxy_label"],
    test_rows_eval["pred_group_adjusted"]
)

perf_df = pd.DataFrame([{
    "AUC (unchanged)": auc,
    "Accuracy (global threshold)": acc_global,
    "Accuracy (group thresholds)": acc_adjusted
}])

print("Performance comparison:")
display(perf_df)

Performance comparison:


,AUC (unchanged),Accuracy (global threshold),Accuracy (group thresholds)
0,0.787263,0.760625,0.760286


In [29]:
# =========================
# Multiple testing correction (FDR)
# =========================

from statsmodels.stats.multitest import multipletests

# AUC tests
auc_tests_df["p_adj_fdr"] = multipletests(
    auc_tests_df["p_value"],
    method="fdr_bh"
)[1]

# Error rate tests (FPR + FNR separately)
error_tests_df["p_adj_fp_fdr"] = multipletests(
    error_tests_df["p_value_fp_rate"],
    method="fdr_bh"
)[1]

error_tests_df["p_adj_fn_fdr"] = multipletests(
    error_tests_df["p_value_fn_rate"],
    method="fdr_bh"
)[1]

print("AUC tests with FDR correction:")
display(auc_tests_df.sort_values("p_adj_fdr"))

print("Error rate tests with FDR correction:")
display(error_tests_df.sort_values("p_adj_fp_fdr"))

AUC tests with FDR correction:


,group_a,group_b,auc_a,auc_b,auc_diff,p_value,p_adj_fdr
3,Female | 45-74,Male | 75+,0.800836,0.770183,0.030653,2.591300e-49,3.886950e-48
2,Female | 45-74,Male | 45-74,0.800836,0.783819,0.017017,6.503777e-43,4.877833e-42
4,Female | 45-74,Male | <45,0.800836,0.778448,0.022388,1.249249e-34,6.246244e-34
0,Female | 45-74,Female | 75+,0.800836,0.784147,0.016690,8.833166e-23,3.312437e-22
12,Male | 45-74,Male | 75+,0.783819,0.770183,0.013636,5.696621e-21,1.708986e-20
10,Female | <45,Male | 75+,0.786947,0.770183,0.016763,8.960066e-14,2.240016e-13
7,Female | 75+,Male | 75+,0.784147,0.770183,0.013963,5.629626e-11,1.206348e-10
1,Female | 45-74,Female | <45,0.800836,0.786947,0.013889,1.396246e-09,2.617961e-09
14,Male | 75+,Male | <45,0.770183,0.778448,-0.008265,2.232488e-07,3.720814e-07
11,Female | <45,Male | <45,0.786947,0.778448,0.008498,1.372292e-04,2.058438e-04


Error rate tests with FDR correction:


,group_a,group_b,fp_rate_a,fp_rate_b,fn_rate_a,fn_rate_b,p_value_fp_rate,p_value_fn_rate,p_adj_fp_fdr,p_adj_fn_fdr
3,Female | 45-74,Male | 75+,0.210542,0.279527,0.299127,0.330097,9.244240e-195,0.478742,1.386636e-193,1.0
4,Female | 45-74,Male | <45,0.210542,0.274450,0.299127,0.266667,2.589232e-131,0.511824,1.941924e-130,1.0
10,Female | <45,Male | 75+,0.210834,0.279527,0.318182,0.330097,5.288112e-105,0.949181,2.644056e-104,1.0
2,Female | 45-74,Male | 45-74,0.210542,0.246916,0.299127,0.322327,4.766451e-100,0.452916,1.787419e-99,1.0
11,Female | <45,Male | <45,0.210834,0.274450,0.318182,0.266667,4.775557e-79,0.483537,1.432667e-78,1.0
7,Female | 75+,Male | 75+,0.229125,0.279527,0.314815,0.330097,3.846145e-74,0.816858,9.615363e-74,1.0
8,Female | 75+,Male | <45,0.229125,0.274450,0.314815,0.266667,2.496835e-50,0.380618,5.350361e-50,1.0
12,Male | 45-74,Male | 75+,0.246916,0.279527,0.322327,0.330097,1.400494e-46,0.903349,2.625927e-46,1.0
9,Female | <45,Male | 45-74,0.210834,0.246916,0.318182,0.322327,1.758665e-41,1.000000,2.931108e-41,1.0
13,Male | 45-74,Male | <45,0.246916,0.274450,0.322327,0.266667,8.112162e-26,0.220236,1.216824e-25,1.0


In [30]:
# =========================
# Effect size summary (practical significance)
# =========================

effect_summary = auc_tests_df.copy()

effect_summary["abs_auc_diff"] = effect_summary["auc_diff"].abs()

# Categorize effect sizes
def categorize_effect(x):
    if x < 0.01:
        return "negligible"
    elif x < 0.02:
        return "small"
    elif x < 0.05:
        return "moderate"
    else:
        return "large"

effect_summary["effect_size"] = effect_summary["abs_auc_diff"].apply(categorize_effect)

print("AUC effect size interpretation:")
display(effect_summary.sort_values("abs_auc_diff", ascending=False))

AUC effect size interpretation:


,group_a,group_b,auc_a,auc_b,auc_diff,p_value,p_adj_fdr,abs_auc_diff,effect_size
3,Female | 45-74,Male | 75+,0.800836,0.770183,0.030653,2.591300e-49,3.886950e-48,0.030653,moderate
4,Female | 45-74,Male | <45,0.800836,0.778448,0.022388,1.249249e-34,6.246244e-34,0.022388,moderate
2,Female | 45-74,Male | 45-74,0.800836,0.783819,0.017017,6.503777e-43,4.877833e-42,0.017017,small
10,Female | <45,Male | 75+,0.786947,0.770183,0.016763,8.960066e-14,2.240016e-13,0.016763,small
0,Female | 45-74,Female | 75+,0.800836,0.784147,0.016690,8.833166e-23,3.312437e-22,0.016690,small
7,Female | 75+,Male | 75+,0.784147,0.770183,0.013963,5.629626e-11,1.206348e-10,0.013963,small
1,Female | 45-74,Female | <45,0.800836,0.786947,0.013889,1.396246e-09,2.617961e-09,0.013889,small
12,Male | 45-74,Male | 75+,0.783819,0.770183,0.013636,5.696621e-21,1.708986e-20,0.013636,small
11,Female | <45,Male | <45,0.786947,0.778448,0.008498,1.372292e-04,2.058438e-04,0.008498,negligible
14,Male | 75+,Male | <45,0.770183,0.778448,-0.008265,2.232488e-07,3.720814e-07,0.008265,negligible


In [31]:
# =========================
# Equalized odds gap
# =========================

comparison["EO_gap_global"] = (
    abs(comparison["FPR_global"] - comparison["FPR_global"].mean()) +
    abs(comparison["FNR_global"] - comparison["FNR_global"].mean())
)

comparison["EO_gap_adjusted"] = (
    abs(comparison["FPR_adjusted"] - comparison["FPR_adjusted"].mean()) +
    abs(comparison["FNR_adjusted"] - comparison["FNR_adjusted"].mean())
)

eo_summary = pd.DataFrame([{
    "EO_gap_global_mean": comparison["EO_gap_global"].mean(),
    "EO_gap_adjusted_mean": comparison["EO_gap_adjusted"].mean()
}])

print("Equalized odds gap summary:")
display(eo_summary)

Equalized odds gap summary:


,EO_gap_global_mean,EO_gap_adjusted_mean
0,0.042158,0.029581


In [ ]:
# =========================
# Plot FPR disparity before vs after
# =========================

plt.figure(figsize=(8, 5))

x = np.arange(len(comparison))
width = 0.35

plt.bar(x - width/2, comparison["FPR_global"], width, label="Global threshold")
plt.bar(x + width/2, comparison["FPR_adjusted"], width, label="Group-adjusted")

plt.xticks(x, comparison["group"], rotation=45)
plt.ylabel("False Positive Rate")
plt.title("FPR by subgroup: before vs after adjustment")
plt.legend()
plt.tight_layout()
plt.show()

In [33]:
# =========================
# Bias attribution summary
# =========================

bias_summary = disparity_gap_df.copy()

bias_summary["delta_auc_gap_vs_full"] = (
    bias_summary["gap_auc"] -
    bias_summary.loc[bias_summary["model"] == "GLM_full", "gap_auc"].values[0]
)

print("Bias attribution (relative to full model):")
display(bias_summary.sort_values("delta_auc_gap_vs_full"))

Bias attribution (relative to full model):


,model,min_auc,max_auc,gap_auc,min_sensitivity_at_optimal_threshold,max_sensitivity_at_optimal_threshold,gap_sensitivity_at_optimal_threshold,min_specificity_at_optimal_threshold,max_specificity_at_optimal_threshold,gap_specificity_at_optimal_threshold,delta_auc_gap_vs_full
0,GLM_full,0.770183,0.800836,0.030653,0.662037,0.740000,0.077963,0.683021,0.833979,0.150957,0.000000
2,GLM_no_age_gender,0.769569,0.800935,0.031366,0.662037,0.733333,0.071296,0.700390,0.835914,0.135525,0.000713
4,GLM_physiology_only,0.683958,0.720825,0.036867,0.580000,0.716157,0.136157,0.620248,0.772085,0.151837,0.006214
1,GLM_no_admin,0.683854,0.721195,0.037342,0.580000,0.731441,0.151441,0.609942,0.771641,0.161700,0.006689
3,GLM_no_missingness,0.703755,0.761916,0.058161,0.606796,0.673333,0.066537,0.663759,0.810148,0.146389,0.027508


In [34]:
# =========================
# Final fairness summary table
# =========================

final_summary = pd.DataFrame([{
    "AUC_overall": shared_row_metrics["GLM_full"]["auc"],
    "FPR_gap_before": summary_df["FPR_gap_global"].iloc[0],
    "FPR_gap_after": summary_df["FPR_gap_adjusted"].iloc[0],
    "FNR_gap_before": summary_df["FNR_gap_global"].iloc[0],
    "FNR_gap_after": summary_df["FNR_gap_adjusted"].iloc[0],
    "Accuracy_global": acc_global,
    "Accuracy_adjusted": acc_adjusted
}])

print("Final fairness summary:")
display(final_summary)

Final fairness summary:


,AUC_overall,FPR_gap_before,FPR_gap_after,FNR_gap_before,FNR_gap_after,Accuracy_global,Accuracy_adjusted
0,0.787263,0.068985,0.006086,0.06343,0.096981,0.760625,0.760286
